# Phase 5 — Name and Address Similarity Matcher

This phase extends the Phase 4 route-based logistic pair scorer with name-token and address-token coverage features. Coverage is the fraction of selected query tokens found in a candidate target record’s indexed postings. Tokens with postings larger than the configured cap are ignored. This is a memory-bounded, asymmetric similarity proxy; it is not full Jaccard or edit distance.

Pairs used for fitting and threshold selection come only from deterministic non-validation Source 1 samples. The fixed Phase 2 holdout is used for final evaluation. No external data is used.

## Results

The model trained on **499,418 candidate pairs** from **15,987 fit queries** (50,827 positives and 448,591 sampled negatives); **4,013 disjoint training queries** selected threshold **0.18874** (calibration macro F0.5 **0.46890**).

On all **441,363** fixed validation queries, macro F0.5 is **0.46791**, compared with Phase 4’s **0.46887** (change **−0.00096**). Micro precision is **21.42%** vs **22.40%**, while micro recall is **48.13%** vs **47.03%**. Candidate recall and candidate coverage are unchanged at **92.14%** and **98.88%**, confirming the comparison uses the same blocking union.

The token-coverage features received small learned weights (name **0.0395**, address **0.0904**). This simple proxy did not improve the target macro metric, so Phase 4 remains the better measured matcher. The full holdout includes singletons, and validation labels were used only for final scoring.

## Run

The full run rebuilds disk-backed indexes for the 10.3 million S2/S3 targets, trains on 20,000 sampled training queries, calibrates on a disjoint train-only slice, and scores all validation queries. Detailed results are saved to `dataset/splits/phase5_similarity_matcher_metrics.json`.

In [ ]:
from pathlib import Path

project_root = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / 'notebooks' / 'validation' / 'phase5_similarity_matcher.py').is_file())
script_path = project_root / 'notebooks' / 'validation' / 'phase5_similarity_matcher.py'
exec(compile(script_path.read_text(encoding='utf-8'), str(script_path), 'exec'), {'__name__': '__main__'})


In [ ]:
import json
from pathlib import Path

project_root = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / 'dataset' / 'splits' / 'phase5_similarity_matcher_metrics.json').is_file())
split_dir = project_root / 'dataset' / 'splits'
phase5 = json.loads((split_dir / 'phase5_similarity_matcher_metrics.json').read_text(encoding='utf-8'))
phase4 = json.loads((split_dir / 'phase4_matcher_metrics.json').read_text(encoding='utf-8'))
for label, result in (('Phase 4 baseline', phase4), ('Phase 5 similarity', phase5)):
    validation = result['validation']
    print('{}: macro F0.5={:.6f}; micro P={:.4f}; micro R={:.4f}'.format(label, validation['macro_f05'], validation['micro_precision'], validation['micro_recall']))
print('Macro F0.5 change: {:+.6f}'.format(phase5['validation']['macro_f05'] - phase4['validation']['macro_f05']))
